# Compare DQN Variants (from training_notes.txt)

Notebook này đọc `logs/<ALGO>/training_notes.txt` để tổng hợp:
- **Training Time** (tổng phút, cộng các lần train/gen)
- **Baseline Avg Time** (Random)
- **Final Avg Time** (final model được chọn)
- **Improve (%)** so với baseline: `Improve = (baseline - final) / baseline * 100`
  - Nếu avg time giảm → Improve dương (cải tiến)
  - Nếu avg time tăng → Improve âm (worse)


In [ ]:
import os
import re
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
LOG_ROOT = os.path.join(ROOT, "logs")

# 6 models trained with generations
ITER_MODELS = ["DQN", "DDQN", "Dueling", "PerDQN", "MultiStepDQN", "Rainbow"]

FOLDER_CANDIDATES = {
    "DQN": ["DQN"],
    "DDQN": ["DDQN"],
    "Dueling": ["Dueling", "DuelDQN", "Duel_DQN", "DuelDQNAgent", "DuelDQN"],
    "PerDQN": ["PerDQN", "PER_DQN", "PERDQN"],
    "MultiStepDQN": ["MultiStepDQN", "MultiStep_DQN", "MultiStep"],
    "Rainbow": ["Rainbow"],
    "FORLAPS": ["FORLAPS"],
    "LearningToAct": ["LearningToAct"],
}

# --- Regex ---
RE_BASELINE_1 = re.compile(r"Random\s*\(Baseline\)\s*:\s*([0-9]+(?:\.[0-9]+)?)", re.IGNORECASE)
RE_BASELINE_2 = re.compile(r"Baseline\s*Avg\s*Time\s*:\s*([0-9]+(?:\.[0-9]+)?)", re.IGNORECASE)
RE_FINAL = re.compile(r"Final Model:\s*(.*?)\s*\(from ep=.*?\)\s*\|\s*Avg Time:\s*([0-9]+(?:\.[0-9]+)?)", re.IGNORECASE)
RE_FINAL_GEN = re.compile(r"gen(\d+)", re.IGNORECASE)
RE_FORLAPS_AVG = re.compile(r"FORLAPS\s*Avg\s*Time\s*:\s*([0-9]+(?:\.[0-9]+)?)", re.IGNORECASE)
RE_LTA_AVG = re.compile(r"LearningToAct\s*Avg\s*Time\s*:\s*([0-9]+(?:\.[0-9]+)?)", re.IGNORECASE)


def find_algo_dir(algo_display_name: str):
    for folder in FOLDER_CANDIDATES.get(algo_display_name, []):
        p = os.path.join(LOG_ROOT, folder)
        if os.path.isdir(p):
            return p
    return None


def read_notes(algo_name: str):
    algo_dir = find_algo_dir(algo_name)
    if not algo_dir:
        return None, None
    note_path = os.path.join(algo_dir, "training_notes.txt")
    if not os.path.exists(note_path):
        return note_path, None
    with open(note_path, "r", encoding="utf-8") as f:
        return note_path, f.read()


def parse_baseline_avg(text: str):
    m = RE_BASELINE_1.search(text)
    if m:
        return float(m.group(1))
    m = RE_BASELINE_2.search(text)
    if m:
        return float(m.group(1))
    return None


def parse_final_by_gen(text: str):
    """Return dict gen->avg_time taken from 'Final Model: ... Avg Time: ...' lines."""
    final_by_gen = {}
    for fname, avg_str in RE_FINAL.findall(text):
        gm = RE_FINAL_GEN.search(fname)
        if not gm:
            continue
        gen = int(gm.group(1))
        final_by_gen[gen] = float(avg_str)
    return final_by_gen


def parse_forlaps_avg(text: str):
    m = RE_FORLAPS_AVG.search(text)
    return float(m.group(1)) if m else None


def parse_lta_avg(text: str):
    m = RE_LTA_AVG.search(text)
    return float(m.group(1)) if m else None


def best_across_gens(final_by_gen: dict):
    if not final_by_gen:
        return None
    return float(min(final_by_gen.values()))


# -----------------------------
# Table 1: Model | Gen1..Gen5
# -----------------------------
GENS = [1, 2, 3, 4, 5]

# Baseline random avg: pick from any available iterative model notes (they all use same baseline)
random_avg = None
for algo in ITER_MODELS:
    _, txt = read_notes(algo)
    if txt:
        random_avg = parse_baseline_avg(txt)
        if random_avg is not None:
            break

rows = []

# Random row
random_row = {"Model": "Random"}
for g in GENS:
    random_row[f"Gen {g}"] = random_avg
rows.append(random_row)

# Iter models rows
for algo in ITER_MODELS:
    _, txt = read_notes(algo)
    final_by_gen = parse_final_by_gen(txt) if txt else {}

    row = {"Model": algo}
    for g in GENS:
        row[f"Gen {g}"] = final_by_gen.get(g, np.nan)
    rows.append(row)


df_by_gen = pd.DataFrame(rows)

# -----------------------------
# Table 2: Overall comparison
# Columns: Random, FORLAPS, LearningToAct, and 6 models (best across gens)
# Rows: Avg, Improve% vs Random
# -----------------------------

# Load FORLAPS / LearningToAct avg from their notes
_, txt_forlaps = read_notes("FORLAPS")
_, txt_lta = read_notes("LearningToAct")

forlaps_avg = parse_forlaps_avg(txt_forlaps) if txt_forlaps else None
lta_avg = parse_lta_avg(txt_lta) if txt_lta else None

# Best across generations for 6 models
best_iter = {}
for algo in ITER_MODELS:
    _, txt = read_notes(algo)
    final_by_gen = parse_final_by_gen(txt) if txt else {}
    best_iter[algo] = best_across_gens(final_by_gen)

cols = ["Random", "FORLAPS", "LearningToAct"] + ITER_MODELS
avg_row = {
    "Random": random_avg,
    "FORLAPS": forlaps_avg,
    "LearningToAct": lta_avg,
}
avg_row.update(best_iter)

impr_row = {}
for k, v in avg_row.items():
    if random_avg is None or random_avg == 0 or v is None or (isinstance(v, float) and np.isnan(v)):
        impr_row[k] = np.nan
    else:
        impr_row[k] = (random_avg - v) / random_avg * 100.0


df_overall = pd.DataFrame([avg_row, impr_row], index=["Avg", "Improve% vs Random"]).reindex(columns=cols)

df_by_gen, df_overall

In [ ]:
# Table 1: 6 models by Gen (Random is repeated across Gen columns)
print('=== Table 1: Iterative models (best per-gen final avg) ===')
display(df_by_gen)

print('\n=== Table 2: Overall comparison (Avg + Improve% vs Random) ===')
display(df_overall)

# Optional: nicer formatting for Improve% row
fmt_overall = df_overall.copy()
for c in fmt_overall.columns:
    if fmt_overall.loc['Improve% vs Random', c] == fmt_overall.loc['Improve% vs Random', c]:
        fmt_overall.loc['Improve% vs Random', c] = f"{fmt_overall.loc['Improve% vs Random', c]:+.2f}%"

print('\n=== Table 2 (formatted) ===')
display(fmt_overall)

In [ ]:
# 1) Improve plot (baseline vs BEST across gens)
plot_df = df_summary[df_summary['NotesFound']].copy()
plot_df = plot_df.dropna(subset=['ImprovePct'])

if len(plot_df) == 0:
    print('Chưa đủ dữ liệu để vẽ Improve(%): không parse được baseline/best từ training_notes.txt.')
else:
    plot_df = plot_df.sort_values('ImprovePct', ascending=False)

    plt.figure(figsize=(10, 4))
    plt.bar(plot_df['Algorithm'], plot_df['ImprovePct'])
    plt.axhline(0, color='black', linewidth=1)
    plt.ylabel('Improve vs Baseline (%)')
    plt.title('So sánh hiệu năng (Best Avg Time) so với Baseline')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


# 2) Training time per Gen plot
if df_train_long.empty:
    print('Không có dữ liệu TrainTime theo Gen để vẽ.')
else:
    max_gen = int(df_train_long['Gen'].max())

    for g in range(1, max_gen + 1):
        sub = df_train_long[df_train_long['Gen'] == g].copy()
        if len(sub) == 0:
            continue

        sub = sub.sort_values('TrainTimeMinutes', ascending=False)

        plt.figure(figsize=(10, 4))
        plt.bar(sub['Algorithm'], sub['TrainTimeMinutes'])
        plt.ylabel('Training Time (minutes)')
        plt.title(f'Training time comparison - Gen {g}')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()